In [6]:
import json
import logging
import uuid
from pathlib import Path

import pendulum
import polars as pl
import requests
import tenacity
import yaml
from sqlalchemy import create_engine, text

In [7]:
from pathlib import Path

project_root = Path("hh_etl_notebook_project")
(project_root / "data").mkdir(parents=True, exist_ok=True)
(project_root / "logs").mkdir(parents=True, exist_ok=True)

config_path = project_root / "config.yaml"

In [8]:
def load_settings(cfg_path: str | Path) -> dict:
    with open(cfg_path, "r", encoding="utf-8") as fh:
        return yaml.safe_load(fh)


CFG = load_settings(config_path)
TABLE_NAME = f'{CFG["database"]["schema"]}.{CFG["database"]["table"]}'


def build_logger(settings: dict, run_id: str) -> logging.LoggerAdapter:
    log_path = project_root / settings["logging"]["file_name"]
    log_path.parent.mkdir(parents=True, exist_ok=True)

    base_logger = logging.getLogger("hh_etl_notebook")
    base_logger.setLevel(getattr(logging, settings["logging"]["level"].upper(), logging.INFO))

    if not base_logger.handlers:
        formatter = logging.Formatter(
            "%(asctime)s | %(levelname)s | etl_id=%(etl_id)s | %(message)s"
        )

        file_handler = logging.FileHandler(log_path, mode="a", encoding="utf-8")
        file_handler.setFormatter(formatter)

        stream_handler = logging.StreamHandler()
        stream_handler.setFormatter(formatter)

        base_logger.addHandler(file_handler)
        base_logger.addHandler(stream_handler)

    return logging.LoggerAdapter(base_logger, extra={"etl_id": run_id})


def db_engine(settings: dict):
    return create_engine(settings["database"]["url"])

In [9]:
def prepare_database(engine, settings: dict) -> None:
    schema_name = settings["database"]["schema"]
    table_name = settings["database"]["table"]

    ddl = f"""
    CREATE SCHEMA IF NOT EXISTS {schema_name};

    CREATE TABLE IF NOT EXISTS {schema_name}.{table_name} (
        vacancy_id BIGINT PRIMARY KEY,
        vacancy_name TEXT,
        employer_name TEXT,
        area_name TEXT,
        published_at TIMESTAMPTZ,
        alternate_url TEXT,
        salary_from NUMERIC,
        salary_to NUMERIC,
        salary_currency TEXT,
        requirement TEXT,
        responsibility TEXT,
        loaded_at TIMESTAMPTZ,
        etl_id TEXT
    );
    """

    with engine.begin() as conn:
        for statement in ddl.strip().split(";"):
            stmt = statement.strip()
            if stmt:
                conn.execute(text(stmt))


In [10]:
def read_hwm(engine) -> str:
    query = f"SELECT MAX(published_at) AS hwm FROM {TABLE_NAME}"
    frame = pl.read_database(query=query, connection=engine)

    if frame.is_empty() or frame["hwm"][0] is None:
        return "2020-01-01T00:00:00+00:00"

    hwm_value = frame["hwm"][0]
    if hasattr(hwm_value, "isoformat"):
        return hwm_value.isoformat()

    return str(hwm_value)


def retry_decorator(settings: dict):
    return tenacity.retry(
        stop=tenacity.stop_after_attempt(settings["retry"]["attempts"]),
        wait=tenacity.wait_fixed(settings["retry"]["wait_seconds"]),
        retry=tenacity.retry_if_exception_type((requests.RequestException, ValueError)),
        reraise=True,
    )


@retry_decorator(CFG)
def fetch_vacancies_page(page_no: int, date_from: str) -> dict:
    api_url = "https://api.hh.ru/vacancies"
    params = {
        "text": CFG["search"]["text"],
        "area": CFG["search"]["area"],
        "per_page": CFG["search"]["per_page"],
        "page": page_no,
        "order_by": CFG["search"]["order_by"],
        "date_from": date_from,
        "only_with_salary": CFG["search"]["only_with_salary"],
    }
    headers = {"User-Agent": CFG["http"]["user_agent"]}

    response = requests.get(
        api_url,
        params=params,
        headers=headers,
        timeout=CFG["http"]["timeout"],
    )
    response.raise_for_status()
    payload = response.json()

    if "items" not in payload:
        raise ValueError("В ответе HH API отсутствует ключ 'items'.")

    return payload

In [11]:
def parse_rows(api_payload: dict, etl_run_id: str) -> list[dict]:
    extracted = []

    for row in api_payload.get("items", []):
        salary_block = row.get("salary") or {}
        snippet = row.get("snippet") or {}

        extracted.append(
            {
                "vacancy_id": int(row["id"]),
                "vacancy_name": row.get("name"),
                "employer_name": (row.get("employer") or {}).get("name"),
                "area_name": (row.get("area") or {}).get("name"),
                "published_at": row.get("published_at"),
                "alternate_url": row.get("alternate_url"),
                "salary_from": salary_block.get("from"),
                "salary_to": salary_block.get("to"),
                "salary_currency": salary_block.get("currency"),
                "requirement": snippet.get("requirement"),
                "responsibility": snippet.get("responsibility"),
                "loaded_at": pendulum.now("UTC").to_iso8601_string(),
                "etl_id": etl_run_id,
            }
        )

    return extracted


def persist_dataframe(dataset: pl.DataFrame, engine, settings: dict, logger) -> None:
    if dataset.is_empty():
        logger.info("Новых записей нет, сохранение не требуется.")
        return

    dataset.write_database(
        table_name=TABLE_NAME,
        connection=engine,
        if_table_exists="append",
    )

    export_dir = project_root / settings["filesystem"]["output_dir"]
    export_dir.mkdir(parents=True, exist_ok=True)

    export_name = f'hh_increment_{pendulum.now("UTC").format("YYYYMMDD_HHmmss")}.parquet'
    export_path = export_dir / export_name

    dataset.write_parquet(
        export_path,
        compression=settings["filesystem"]["compression"],
    )

    logger.info("Сохранено в БД строк: %s", dataset.height)
    logger.info("Сохранено в parquet: %s", export_path)


def run_incremental_etl() -> pl.DataFrame:
    current_etl_id = str(uuid.uuid4())
    logger = build_logger(CFG, current_etl_id)
    logger.info("ETL-запуск начат.")

    engine = db_engine(CFG)
    prepare_database(engine, CFG)

    high_water_mark = read_hwm(engine)
    logger.info("Текущий HWM: %s", high_water_mark)

    collected_rows = []
    pages_to_read = CFG["search"]["pages"]

    for page_idx in range(pages_to_read):
        try:
            page_payload = fetch_vacancies_page(page_idx, high_water_mark)
            page_rows = parse_rows(page_payload, current_etl_id)
            logger.info("Страница %s обработана, получено строк: %s", page_idx, len(page_rows))
            collected_rows.extend(page_rows)
        except Exception as err:
            logger.exception("Ошибка при обработке страницы %s: %s", page_idx, err)
            raise

    if not collected_rows:
        logger.info("Источник не вернул новых данных.")
        return pl.DataFrame()

    batch_df = pl.DataFrame(collected_rows)

    if "published_at" in batch_df.columns:
        batch_df = batch_df.with_columns(
            pl.col("published_at").str.to_datetime(strict=False, time_zone="UTC")
        )

    existing_ids = pl.read_database(
        query=f"SELECT vacancy_id FROM {TABLE_NAME}",
        connection=engine,
    )

    if not existing_ids.is_empty():
        batch_df = batch_df.join(existing_ids, on="vacancy_id", how="anti")

    persist_dataframe(batch_df, engine, CFG, logger)
    logger.info("ETL-запуск завершён. Итоговых новых строк: %s", batch_df.height)

    return batch_df

In [12]:
result_df = run_incremental_etl()
result_df.head()

2026-03-31 21:56:40,323 | INFO | etl_id=223377a3-4466-4031-bbca-8ee1dd8e6848 | ETL-запуск начат.
2026-03-31 21:56:40,422 | INFO | etl_id=223377a3-4466-4031-bbca-8ee1dd8e6848 | Текущий HWM: 2026-03-31T17:22:28+00:00
2026-03-31 21:56:40,776 | INFO | etl_id=223377a3-4466-4031-bbca-8ee1dd8e6848 | Страница 0 обработана, получено строк: 1
2026-03-31 21:56:40,888 | INFO | etl_id=223377a3-4466-4031-bbca-8ee1dd8e6848 | Страница 1 обработана, получено строк: 0
2026-03-31 21:56:41,033 | INFO | etl_id=223377a3-4466-4031-bbca-8ee1dd8e6848 | Страница 2 обработана, получено строк: 0
2026-03-31 21:56:41,210 | INFO | etl_id=223377a3-4466-4031-bbca-8ee1dd8e6848 | Страница 3 обработана, получено строк: 0
2026-03-31 21:56:41,220 | INFO | etl_id=223377a3-4466-4031-bbca-8ee1dd8e6848 | Новых записей нет, сохранение не требуется.
2026-03-31 21:56:41,221 | INFO | etl_id=223377a3-4466-4031-bbca-8ee1dd8e6848 | ETL-запуск завершён. Итоговых новых строк: 0


vacancy_id,vacancy_name,employer_name,area_name,published_at,alternate_url,salary_from,salary_to,salary_currency,requirement,responsibility,loaded_at,etl_id
i64,str,str,str,"datetime[μs, UTC]",str,null,null,null,str,str,str,str
